# SDF 01 -- Signed Distance Fields: Implicit Geometry & Marching Cubes

> **Geo-Instructor . SDF/Geometry Career Track . Notebook 1 of 3**

An SDF is a scalar field f(x) = signed distance to the nearest surface.
Zero level set = the surface. Negative = inside. Positive = outside.
This single representation unifies geometry, collision, rendering, and ML.

```
Runtime: ~5 min  |  No GPU  |  numpy + matplotlib only
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
np.random.seed(0)
plt.rcParams.update({'figure.facecolor':'#F4F2F0','axes.facecolor':'#F4F2F0','font.family':'monospace'})
print('Ready.')

---
## Part 1 -- SDF Primitives in 2D

The key formulas (all in 2D first):
```
  Circle:   f(p) = |p - center| - r
  Box:      f(p) = |max(|p| - b, 0)| - r       (rounded box with radius r)
  Segment:  project p onto segment, return dist to projection
  Capsule:  segment SDF - radius
```
The zero contour of each function is the shape boundary.

In [ ]:
def sdf_circle(px, py, cx, cy, r):
    return np.sqrt((px-cx)**2 + (py-cy)**2) - r

def sdf_box2d(px, py, bx, by):
    qx = np.abs(px) - bx; qy = np.abs(py) - by
    return np.sqrt(np.maximum(qx,0)**2 + np.maximum(qy,0)**2) + np.minimum(np.maximum(qx,qy), 0)

def sdf_capsule2d(px, py, ax, ay, bx, by, r):
    pax,pay = px-ax, py-ay; babx,baby = bx-ax, by-ay
    h = np.clip((pax*babx + pay*baby) / (babx**2+baby**2+1e-10), 0, 1)
    dx = pax - h*babx; dy = pay - h*baby
    return np.sqrt(dx**2+dy**2) - r

N = 300
x = np.linspace(-2, 2, N); y = np.linspace(-2, 2, N)
X, Y = np.meshgrid(x, y)

sdfs = [
    ('Circle', sdf_circle(X, Y, 0, 0, 1.0)),
    ('Rounded Box', sdf_box2d(X, Y, 0.7, 0.5)),
    ('Capsule', sdf_capsule2d(X, Y, -0.8, -0.4, 0.8, 0.4, 0.35)),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('SDF Primitives -- zero level set = surface', fontsize=11)
for ax, (title, sdf) in zip(axes, sdfs):
    im = ax.contourf(X, Y, sdf, levels=30, cmap='RdBu_r', vmin=-1.5, vmax=1.5)
    ax.contour(X, Y, sdf, levels=[0], colors='k', linewidths=2.5)
    ax.contour(X, Y, sdf, levels=np.linspace(-1.4, 1.4, 12), colors='k', linewidths=0.4, alpha=0.3)
    ax.set_title(title); ax.set_aspect('equal'); plt.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout(); plt.show()
print('Blue = inside (negative), Red = outside (positive), Black contour = surface (zero).')

---
## Part 2 -- CSG Boolean Operations

The most powerful feature of SDFs: Boolean ops are just `min`/`max`.
```
  Union:        f(p) = min(f1(p), f2(p))
  Intersection: f(p) = max(f1(p), f2(p))
  Subtraction:  f(p) = max(f1(p), -f2(p))
  Smooth union: smin(f1, f2, k) -- polynomial blend controlled by k
```

In [ ]:
def smin(a, b, k):
    """Smooth minimum (polynomial version by Inigo Quilez)."""
    h = np.clip(0.5 + 0.5*(b-a)/k, 0, 1)
    return a + h*(b-a) - k*h*(1-h)

f1 = sdf_circle(X, Y, -0.4, 0, 0.8)
f2 = sdf_circle(X, Y,  0.4, 0, 0.8)
ops = [
    ('Union\nmin(f1,f2)', np.minimum(f1, f2)),
    ('Intersection\nmax(f1,f2)', np.maximum(f1, f2)),
    ('Subtraction\nmax(f1,-f2)', np.maximum(f1, -f2)),
    ('Smooth Union\nsmin(f1,f2,k=0.5)', smin(f1, f2, 0.5)),
]
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5))
fig.suptitle('SDF Boolean Operations (CSG)', fontsize=11)
for ax, (title, sdf) in zip(axes, ops):
    ax.contourf(X, Y, sdf, levels=30, cmap='RdBu_r', vmin=-1.5, vmax=1.5)
    ax.contour(X, Y, sdf, levels=[0], colors='k', linewidths=2.5)
    ax.set_title(title, fontsize=9); ax.set_aspect('equal')
plt.tight_layout(); plt.show()
print('smin produces organic, smooth blending between shapes -- used in character modeling.')

---
## Part 3 -- 3D SDF & the Eikonal Property

All the same formulas extend to 3D. The gradient of an SDF has a special property:
```
  |grad f(p)| = 1   everywhere (the eikonal equation)
```
This means: the gradient is always a unit vector = the surface normal.

In [ ]:
def sdf_sphere3d(px, py, pz, r): return np.sqrt(px**2+py**2+pz**2) - r
def sdf_box3d(px, py, pz, bx, by, bz):
    qx=np.abs(px)-bx; qy=np.abs(py)-by; qz=np.abs(pz)-bz
    return np.sqrt(np.maximum(qx,0)**2+np.maximum(qy,0)**2+np.maximum(qz,0)**2) + np.minimum(np.maximum(qx,np.maximum(qy,qz)),0)
def sdf_torus3d(px, py, pz, R, r):
    q = np.sqrt(px**2+pz**2) - R
    return np.sqrt(q**2+py**2) - r

N3 = 40
x3 = np.linspace(-2, 2, N3); y3 = np.linspace(-2, 2, N3); z3 = np.linspace(-2, 2, N3)
X3, Y3, Z3 = np.meshgrid(x3, y3, z3, indexing='ij')

sdf_sp = sdf_sphere3d(X3, Y3, Z3, 1.0)

# Verify eikonal: compute gradient numerically
dx = x3[1]-x3[0]
gx = np.gradient(sdf_sp, dx, axis=0)
gy = np.gradient(sdf_sp, dx, axis=1)
gz = np.gradient(sdf_sp, dx, axis=2)
grad_norm = np.sqrt(gx**2 + gy**2 + gz**2)

# Sample near surface
near_surface = np.abs(sdf_sp) < 0.15
if near_surface.sum() > 0:
    norms_near = grad_norm[near_surface]
    norms_bulk = grad_norm[~near_surface]
    print(f'Gradient norm near surface:  mean={norms_near.mean():.4f}  std={norms_near.std():.4f} (should be ~1.0)')
    print(f'Gradient norm in bulk:       mean={norms_bulk.mean():.4f}  std={norms_bulk.std():.4f}')

# Show slices
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('3D SDF slices (z=0) and eikonal property', fontsize=10)
mid = N3 // 2
for ax, (f3d, title) in zip(axes, [
    (sdf_sp, 'Sphere'),
    (sdf_box3d(X3,Y3,Z3,0.8,0.5,0.6), 'Box'),
    (sdf_torus3d(X3,Y3,Z3,0.8,0.3), 'Torus')]):
    im = ax.contourf(X3[:,mid,:], Z3[:,mid,:], f3d[:,mid,:], levels=25, cmap='RdBu_r', vmin=-1.5, vmax=1.5)
    ax.contour(X3[:,mid,:], Z3[:,mid,:], f3d[:,mid,:], levels=[0], colors='k', linewidths=2)
    ax.set_title(title); ax.set_aspect('equal')
plt.tight_layout(); plt.show()

---
## Part 4 -- Marching Cubes

Given a 3D SDF grid, extract the zero-level-set as a triangle mesh.
The key idea: classify each of the 8 cube vertices as inside (SDF<0) or outside (SDF>0).
The 2^8 = 256 configurations reduce to 15 unique cases via symmetry.
Each case has a fixed triangulation from the edge table.

In [ ]:
# Minimal Marching Cubes implementation
# Edge table: for each of 256 vertex configs, which edges are cut
EDGE_TABLE = [
    0x0,0x109,0x203,0x30a,0x406,0x50f,0x605,0x70c,0x80c,0x905,0xa0f,0xb06,0xc0a,0xd03,0xe09,0xf00,
    0x190,0x99,0x393,0x29a,0x596,0x49f,0x795,0x69c,0x99c,0x895,0xb9f,0xa96,0xd9a,0xc93,0xf99,0xe90,
    0x230,0x339,0x33,0x13a,0x636,0x73f,0x435,0x53c,0xa3c,0xb35,0x83f,0x936,0xe3a,0xf33,0xc39,0xd30,
    0x3a0,0x2a9,0x1a3,0xaa,0x7a6,0x6af,0x5a5,0x4ac,0xbac,0xaa5,0x9af,0x8a6,0xfaa,0xea3,0xda9,0xca0,
    0x460,0x569,0x663,0x76a,0x66,0x16f,0x265,0x36c,0xc6c,0xd65,0xe6f,0xf66,0x86a,0x963,0xa69,0xb60,
    0x5f0,0x4f9,0x7f3,0x6fa,0x1f6,0xff,0x3f5,0x2fc,0xdfc,0xcf5,0xfff,0xef6,0x9fa,0x8f3,0xbf9,0xaf0,
    0x650,0x759,0x453,0x55a,0x256,0x35f,0x55,0x15c,0xe5c,0xf55,0xc5f,0xd56,0xa5a,0xb53,0x859,0x950,
    0x7c0,0x6c9,0x5c3,0x4ca,0x3c6,0x2cf,0x1c5,0xcc,0xfcc,0xec5,0xdcf,0xcc6,0xbca,0xac3,0x9c9,0x8c0,
    0x8c0,0x9c9,0xac3,0xbca,0xcc6,0xdcf,0xec5,0xfcc,0xcc,0x1c5,0x2cf,0x3c6,0x4ca,0x5c3,0x6c9,0x7c0,
    0x950,0x859,0xb53,0xa5a,0xd56,0xc5f,0xf55,0xe5c,0x15c,0x55,0x35f,0x256,0x55a,0x453,0x759,0x650,
    0xaf0,0xbf9,0x8f3,0x9fa,0xef6,0xfff,0xcf5,0xdfc,0x2fc,0x3f5,0xff,0x1f6,0x6fa,0x7f3,0x4f9,0x5f0,
    0xb60,0xa69,0x963,0x86a,0xf66,0xe6f,0xd65,0xc6c,0x36c,0x265,0x16f,0x66,0x76a,0x663,0x569,0x460,
    0xca0,0xda9,0xea3,0xfaa,0x8a6,0x9af,0xaa5,0xbac,0x4ac,0x5a5,0x6af,0x7a6,0xaa,0x1a3,0x2a9,0x3a0,
    0xd30,0xc39,0xf33,0xe3a,0x936,0x835,0xb3f,0xa36,0x53c,0x435,0x73f,0x636,0x13a,0x33,0x339,0x230,
    0xe90,0xf99,0xc93,0xd9a,0xa96,0xb9f,0x895,0x99c,0x69c,0x795,0x49f,0x596,0x29a,0x393,0x99,0x190,
    0xf00,0xe09,0xd03,0xc0a,0xb06,0xa0f,0x905,0x80c,0x70c,0x605,0x50f,0x406,0x30a,0x203,0x109,0x0
]
TRI_TABLE = [
    [],[-1,8,3],[],[8,-1,0],[],[],[],[],[2,3,11],[],[],[],[],[],[-1,2,11,8],[],
    [4,5,9],[],[0,5,9],[],[],[9,5,4],[],[],[],[],[],[],[],[],[],
    [],[4,7,8],[],[],[],[],[],[3,0,8,4,7],[],[],[],[],[],[],[],[],
    [4,6,10,5,4,7,6,5],[],[5,9,4,8,4,7],[],[],[],[],[],[],[],[],[],[],[],[],
    [9,10,11],[2,0,11,11,0,9,9,0,10],[],[11,3,0,9,10,11,0,10,9],[],[],[],[],[],[],[],[],[],[],[],
    [],[0,1,9],[],[],[],[],[],[],[],[],[],[],[],[],[],[],
    [2,1,10],[],[0,1,2,0,2,8,8,2,10],[],[],[],[],[],[],[],[],[],[],[],[],[],
    [10,6,5,0,8,3],[],[10,5,0,10,0,3,10,3,6,11,6,3],[],[],[],[],[],[],[],[],[],[],[],[],[],
    [],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],
    [],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],
    [],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],
    [],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],
    [],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],
    [],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],
    [],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],
    [],[],[],[],[],[],[],[],[],[],[],[],[],[],[],[],
]

def interp_vertex(iso, p1, p2, v1, v2):
    if abs(v1 - v2) < 1e-10: return p1
    t = (iso - v1) / (v2 - v1)
    return p1 + t * (p2 - p1)

def marching_cubes_simple(sdf_func, N=30, bound=1.5, iso=0.0):
    """Simple marching cubes for any SDF function. Returns list of triangles."""
    xs = np.linspace(-bound, bound, N)
    dx = xs[1] - xs[0]
    # Evaluate SDF on grid
    X, Y, Z = np.meshgrid(xs, xs, xs, indexing='ij')
    F = sdf_func(X, Y, Z)
    triangles = []
    # Cube corner offsets (0..7)
    offsets = np.array([[0,0,0],[1,0,0],[1,1,0],[0,1,0],[0,0,1],[1,0,1],[1,1,1],[0,1,1]])
    # Edge pairs (vertex indices)
    edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
    for i in range(N-1):
        for j in range(N-1):
            for k in range(N-1):
                corner_vals = np.array([F[i+o[0], j+o[1], k+o[2]] for o in offsets])
                corner_pts  = np.array([xs[[i+o[0], j+o[1], k+o[2]]] for o in offsets])
                cube_idx = sum((1<<b) for b in range(8) if corner_vals[b] < iso)
                if EDGE_TABLE[cube_idx] == 0: continue
                verts = {}
                for e, (a, b) in enumerate(edges):
                    if EDGE_TABLE[cube_idx] & (1<<e):
                        verts[e] = interp_vertex(iso, corner_pts[a], corner_pts[b], corner_vals[a], corner_vals[b])
                tris = TRI_TABLE[cube_idx]
                for t in range(0, len(tris)-2, 3):
                    tri = [tris[t], tris[t+1], tris[t+2]]
                    if all(v in verts for v in tri):
                        triangles.append([verts[v] for v in tri])
    return triangles

print('Building SDF grids and running Marching Cubes...')
results = []
for name, fn in [
    ('Sphere', lambda x,y,z: np.sqrt(x**2+y**2+z**2)-1.0),
    ('Torus',  lambda x,y,z: np.sqrt((np.sqrt(x**2+z**2)-0.9)**2+y**2)-0.35),
]:
    tris = marching_cubes_simple(fn, N=28)
    results.append((name, tris))
    print(f'  {name}: {len(tris)} triangles')

fig = plt.figure(figsize=(12, 5))
fig.suptitle('Marching Cubes -- zero level set extraction', fontsize=11)
for i, (name, tris) in enumerate(results):
    ax = fig.add_subplot(1, 2, i+1, projection='3d')
    if tris:
        poly = Poly3DCollection(tris, alpha=0.6, facecolor='steelblue', edgecolor='none')
        ax.add_collection3d(poly)
    ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_zlim(-1.5,1.5)
    ax.set_title(f'{name} ({len(tris)} tris)')
plt.tight_layout(); plt.show()

---
## Exercises

### Exercise 1 -- Shell operation
`shell(f, thickness) = abs(f) - thickness/2` -- hollow out any solid. Try it on the sphere.

### Exercise 2 -- Domain repetition
`f(mod(p, cell_size) - cell_size/2)` tiles any SDF infinitely. Build a grid of spheres.

### Exercise 3 -- SDF from mesh
Given a triangle mesh, compute the SDF by: for each grid point, find the closest point on any triangle, measure distance, set sign by checking if point is inside or outside using ray casting.

---
## Summary

| Concept | Formula | Library |
|---------|---------|--------|
| Sphere SDF | `|p| - r` | OpenVDB, libigl |
| Union | `min(f1, f2)` | Any SDF library |
| Smooth union | `smin(f1, f2, k)` | Shadertoy, Blender |
| Marching Cubes | 256-case lookup | scikit-image `marching_cubes` |
| Dual Contouring | sharper features than MC | OpenVDB DC |
| Neural SDF | MLP learns f(p) | DeepSDF, NeuS |

In [ ]:
print('Notebook complete.')
print('Key takeaways:')
print('  SDF f(p)         -> scalar field, zero = surface, sign = inside/outside')
print('  CSG via min/max  -> Boolean operations in one line')
print('  Eikonal property -> gradient is always unit normal')
print('  Marching Cubes   -> extract mesh from any implicit function')
print()
print('Production: OpenVDB (Houdini, Blender), libigl, CGAL, SDF shader in Shadertoy')